# 🚀 Ollama GPU Server (Kaggle)

This notebook starts an Ollama server on Kaggle's free GPU and exposes it
via ngrok for remote access by your AI TDD Orchestrator pipeline.

## Setup
1. Settings → Accelerator → **GPU T4 x2**
2. Set your ngrok auth token below
3. Run all cells
4. Copy the `KAGGLE_OLLAMA_URL` output

## Free Tier Limits
- GPU: NVIDIA P100/T4 (16GB VRAM)
- **30 GPU hours per week** (resets weekly)
- Monitor at: https://www.kaggle.com/me/account → GPU Quota

In [ ]:
# ============================================
# CONFIGURATION
# ============================================
NGROK_AUTH_TOKEN = ""  # Get free at https://dashboard.ngrok.com/get-started/your-authtoken
OLLAMA_MODEL = "qwen2.5-coder:7b"

# Verify GPU
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print(f'🎮 GPU: {result.stdout.strip()}')
assert 'NVIDIA' in result.stdout, '❌ No GPU! Enable in Settings → Accelerator → GPU T4 x2'

In [ ]:
# ============================================
# Step 1: Install Ollama
# ============================================
!curl -fsSL https://ollama.com/install.sh | sh
print('✅ Ollama installed')

In [ ]:
# ============================================
# Step 2: Start Ollama server
# ============================================
import subprocess, time, requests

proc = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print(f'🔄 Ollama starting (PID: {proc.pid})')

for i in range(30):
    try:
        if requests.get('http://localhost:11434/api/tags', timeout=2).status_code == 200:
            print('✅ Ollama is ready!')
            break
    except: pass
    time.sleep(2)
else:
    print('❌ Ollama failed to start')

In [ ]:
# ============================================
# Step 3: Pull model
# ============================================
!ollama pull {OLLAMA_MODEL}
print(f'✅ Model {OLLAMA_MODEL} ready')

In [ ]:
# ============================================
# Step 4: Expose via ngrok
# ============================================
!pip install pyngrok -q
from pyngrok import ngrok

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

tunnel = ngrok.connect(11434)
public_url = tunnel.public_url

print('\n' + '='*60)
print('🎉 OLLAMA IS LIVE ON KAGGLE!')
print('='*60)
print(f'\n📡 Public URL: {public_url}')
print(f'\n🔧 Set environment variable:')
print(f'   export KAGGLE_OLLAMA_URL="{public_url}"')
print(f'\n🔧 Or add to GitHub Secrets:')
print(f'   KAGGLE_OLLAMA_URL = {public_url}')
print(f'\n⏱️  Kaggle quota: 30 GPU hrs/week')
print('='*60)

In [ ]:
# ============================================
# Step 5: Keep alive
# ============================================
import time
print('🔄 Keep-alive running. Do not close this tab.')

uptime = 0
while True:
    time.sleep(60)
    uptime += 1
    try: requests.get('http://localhost:11434/api/tags', timeout=5)
    except: pass
    if uptime % 10 == 0:
        print(f'⏱️  Uptime: {uptime} min | URL: {public_url}')